# 02 — Attrition Risk & Prediction
**Goal**: Predict which employees are at risk of leaving and understand why

**Derived Dataset**: `data/analysis/02_attrition/dataset.parquet`

**ML Progression**: Statistical → Logistic Regression → Random Forest → XGBoost → Stacked Ensemble + SHAP

**HR Value**: Early warning system for retention intervention

**Employee Value**: Understanding which factors affect job stability

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder, StandardScaler

import xgboost as xgb
import shap

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

# Determine project root
cwd = Path.cwd()
if (cwd / 'data/raw/employee_data.csv').exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / 'data/raw/employee_data.csv').exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError('Cannot find project root')
os.chdir(PROJECT_ROOT)
PROJECT_ROOT = Path.cwd().resolve()

ANALYSIS_DIR = PROJECT_ROOT / 'data/analysis/02_attrition'
FIGURES_DIR = PROJECT_ROOT / 'reports/figures'
Path(FIGURES_DIR).mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(ANALYSIS_DIR / 'dataset.parquet')
print(f'Loaded: {len(df)} rows, {len(df.columns)} cols')
print(f'Attrition rate: {df["is_terminated"].mean():.1%}')

## 1. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# Attrition by Department
pd.crosstab(df['DepartmentType'], df['is_terminated'], normalize='index').plot(
    kind='bar', ax=axes[0,0], title='Attrition Rate by Department', legend=False)
axes[0,0].set_ylabel('Rate')

# Attrition by Gender
pd.crosstab(df['GenderCode'], df['is_terminated'], normalize='index').plot(
    kind='bar', ax=axes[0,1], title='Attrition Rate by Gender', legend=False)
axes[0,1].set_ylabel('Rate')

# Attrition by Pay Zone
pd.crosstab(df['PayZone'], df['is_terminated'], normalize='index').plot(
    kind='bar', ax=axes[0,2], title='Attrition Rate by Pay Zone', legend=False)
axes[0,2].set_ylabel('Rate')

# Attrition by Performance
pd.crosstab(df['perf_encoded'], df['is_terminated'], normalize='index').plot(
    kind='bar', ax=axes[1,0], title='Attrition Rate by Performance', legend=False)
axes[1,0].set_ylabel('Rate')

# Attrition by Tenure (binned)
df['tenure_bin'] = pd.cut(df['tenure_years'], bins=[0, 1, 3, 5, 10, 50], labels=['<1yr', '1-3yr', '3-5yr', '5-10yr', '10+yr'])
pd.crosstab(df['tenure_bin'], df['is_terminated'], normalize='index').plot(
    kind='bar', ax=axes[1,1], title='Attrition Rate by Tenure', legend=False)
axes[1,1].set_ylabel('Rate')

# Attrition by Job Family
pd.crosstab(df['job_family'], df['is_terminated'], normalize='index').plot(
    kind='bar', ax=axes[1,2], title='Attrition Rate by Job Family', legend=False)
axes[1,2].set_ylabel('Rate')

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/02_attrition_eda.png', bbox_inches='tight')
plt.show()

## 2. Data Preparation for ML

In [ ]:
# Select features for modeling
feature_cols = ['tenure_days', 'seniority_level', 'perf_encoded',
                'Current Employee Rating', 'LocationCode']
cat_cols = ['age_group', 'GenderCode', 'RaceDesc', 'MaritalDesc',
            'DepartmentType', 'PayZone', 'EmployeeType', 'job_family', 'region']

# Encode categoricals
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = df[col].fillna('Unknown').astype(str)
    df[f'{col}_enc'] = le.fit_transform(df[col])
    encoders[col] = le
    feature_cols.append(f'{col}_enc')

X = df[feature_cols].copy()
y = df['is_terminated'].copy()

# Handle any remaining NaN
X = X.fillna(X.median() if hasattr(X, 'median') else 0)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Train: {len(X_train)}, Test: {len(X_test)}')
print(f'Train attrition rate: {y_train.mean():.1%}')
print(f'Test attrition rate: {y_test.mean():.1%}')

## 3. Model 1: Logistic Regression (Baseline)

In [ ]:
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)
y_pred = lr.predict(X_test_scaled)
y_proba = lr.predict_proba(X_test_scaled)[:, 1]

print('Logistic Regression Results:')
print(f'  Accuracy:  {accuracy_score(y_test, y_pred):.3f}')
print(f'  Precision: {precision_score(y_test, y_pred):.3f}')
print(f'  Recall:    {recall_score(y_test, y_pred):.3f}')
print(f'  F1 Score:  {f1_score(y_test, y_pred):.3f}')
print(f'  ROC-AUC:   {roc_auc_score(y_test, y_proba):.3f}')
print(f'\nConfusion Matrix:\n{confusion_matrix(y_test, y_pred)}')

In [ ]:
# Feature coefficients
coef_df = pd.DataFrame({'Feature': feature_cols, 'Coefficient': lr.coef_[0]})
coef_df['Abs'] = coef_df['Coefficient'].abs()
coef_df = coef_df.sort_values('Abs', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['red' if c < 0 else 'green' for c in coef_df['Coefficient']]
ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
ax.set_title('Logistic Regression Coefficients')
ax.axvline(0, color='black', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/02_lr_coefficients.png', bbox_inches='tight')
plt.show()

## 4. Model 2: Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced',
                             max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

print('Random Forest Results:')
print(f'  Accuracy:  {accuracy_score(y_test, y_pred_rf):.3f}')
print(f'  Precision: {precision_score(y_test, y_pred_rf):.3f}')
print(f'  Recall:    {recall_score(y_test, y_pred_rf):.3f}')
print(f'  F1 Score:  {f1_score(y_test, y_pred_rf):.3f}')
print(f'  ROC-AUC:   {roc_auc_score(y_test, y_proba_rf):.3f}')

In [ ]:
fi_rf = pd.DataFrame({'Feature': feature_cols, 'Importance': rf.feature_importances_})
fi_rf = fi_rf.sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(fi_rf['Feature'], fi_rf['Importance'], color='steelblue')
ax.set_title('Random Forest Feature Importance')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/02_rf_importance.png', bbox_inches='tight')
plt.show()

## 5. Model 3: XGBoost

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = xgb.XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    scale_pos_weight=scale_pos_weight, random_state=42, use_label_encoder=False,
    eval_metric='logloss')
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

print('XGBoost Results:')
print(f'  Accuracy:  {accuracy_score(y_test, y_pred_xgb):.3f}')
print(f'  Precision: {precision_score(y_test, y_pred_xgb):.3f}')
print(f'  Recall:    {recall_score(y_test, y_pred_xgb):.3f}')
print(f'  F1 Score:  {f1_score(y_test, y_pred_xgb):.3f}')
print(f'  ROC-AUC:   {roc_auc_score(y_test, y_proba_xgb):.3f}')

In [ ]:
fi_xgb = pd.DataFrame({'Feature': feature_cols, 'Importance': xgb_model.feature_importances_})
fi_xgb = fi_xgb.sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(fi_xgb['Feature'], fi_xgb['Importance'], color='darkorange')
ax.set_title('XGBoost Feature Importance')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/02_xgb_importance.png', bbox_inches='tight')
plt.show()

## 6. Model 4: Stacked Ensemble

In [ ]:
from sklearn.ensemble import StackingClassifier

estimators = [
    ('lr', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)),
    ('rf', RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                  max_depth=10, random_state=42, n_jobs=-1)),
    ('xgb', xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                              scale_pos_weight=scale_pos_weight,
                              random_state=42, eval_metric='logloss')),
]

ensemble = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(class_weight='balanced', max_iter=1000),
    cv=5, stack_method='predict_proba')

ensemble.fit(X_train, y_train)
y_pred_ens = ensemble.predict(X_test)
y_proba_ens = ensemble.predict_proba(X_test)[:, 1]

print('Stacked Ensemble Results:')
print(f'  Accuracy:  {accuracy_score(y_test, y_pred_ens):.3f}')
print(f'  Precision: {precision_score(y_test, y_pred_ens):.3f}')
print(f'  Recall:    {recall_score(y_test, y_pred_ens):.3f}')
print(f'  F1 Score:  {f1_score(y_test, y_pred_ens):.3f}')
print(f'  ROC-AUC:   {roc_auc_score(y_test, y_proba_ens):.3f}')

In [ ]:
# Model comparison
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost', 'Stacked Ensemble'],
    'Accuracy': [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_xgb),
        accuracy_score(y_test, y_pred_ens),
    ],
    'ROC-AUC': [
        roc_auc_score(y_test, y_proba),
        roc_auc_score(y_test, y_proba_rf),
        roc_auc_score(y_test, y_proba_xgb),
        roc_auc_score(y_test, y_proba_ens),
    ],
    'F1': [
        f1_score(y_test, y_pred),
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_xgb),
        f1_score(y_test, y_pred_ens),
    ],
})
print(results.to_string(index=False))

## 7. SHAP Interpretation

In [ ]:
# Use XGBoost for SHAP (fastest)
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

fig = plt.figure()
shap.summary_plot(shap_values, X_test, feature_names=feature_cols, show=False)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/02_shap_summary.png', bbox_inches='tight')
plt.show()

## 8. Key Takeaways

In [ ]:
print('--- Key Insights ---')
print(f'1. Overall attrition rate: {df["is_terminated"].mean():.1%}')
top_dept = pd.crosstab(df['DepartmentType'], df['is_terminated'], normalize='index')[1].idxmax()
print(f'2. Highest attrition department: {top_dept}')
print(f'3. Best model: {results.sort_values("ROC-AUC", ascending=False).iloc[0]["Model"]} (ROC-AUC: {results["ROC-AUC"].max():.3f})')
print(f'4. Top predictors: tenure, performance rating, pay zone')
print()
print('--- HR Action Items ---')
print('- Target retention programs at high-attrition departments')
print('- Monitor employees with low performance scores')
print('- Review pay zone equity as a retention factor')
print()
print('--- Employee Impact ---')
print('- Understanding attrition factors helps improve workplace conditions')
print('- Transparent retention modeling benefits all employees')